In [3]:
import pandas as pd
pd.set_option("display.max_columns", 50)

amr = pd.read_csv("../data/raw/amr_ecoli.tsv", sep="\t", dtype=str)
print(amr.shape)
amr.head()

(489375, 10)


,genome_id,genome_name,taxon_id,antibiotic,resistant_phenotype,measurement,measurement_value,measurement_unit,laboratory_typing_method,computational_method
0,562.193047,Escherichia coli ECBSAC210134367,562,ciprofloxacin,Resistant,NaN,NaN,NaN,NaN,SIR XGBoost Model
1,562.227303,Escherichia coli PNUSAE191092,562,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,SIR XGBoost Model
2,562.66875,Escherichia coli strain JS316,562,ciprofloxacin,NaN,8.0,8.0,mg/L,NaN,MIC XGBoost Model
3,562.36249,Escherichia coli strain CP53,562,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,SIR XGBoost Model
4,562.205012,Escherichia coli PNUSAE009306,562,ciprofloxacin,NaN,0.25,0.25,mg/L,NaN,MIC XGBoost Model


In [4]:
#antibiotice si genome
print(amr["genome_id"].nunique(), "genome distincte")
amr["antibiotic"].value_counts()

114985 genome distincte


antibiotic
ciprofloxacin    246737
ampicillin       242638
Name: count, dtype: int64

In [5]:
#lab sau predictie?
print(amr["laboratory_typing_method"].value_counts(dropna=False))
print(amr["computational_method"].value_counts(dropna=False))

laboratory_typing_method
NaN               460567
Disk diffusion     13349
Broth dilution      8130
MIC                 6235
Agar dilution       1094
Name: count, dtype: int64
computational_method
SIR XGBoost Model    229796
MIC XGBoost Model    229734
NaN                   29845
Name: count, dtype: int64


In [6]:
#doar rezultate de laborator
lab = amr[amr["computational_method"].isna()].copy()
print(len(amr), "->", len(lab), "randuri dupa excluderea predictiilor")
print(lab.groupby("antibiotic")["resistant_phenotype"].value_counts())

489375 -> 29845 randuri dupa excluderea predictiilor
antibiotic     resistant_phenotype
ampicillin     Resistant              4546
               Susceptible            3008
               Intermediate             14
ciprofloxacin  Susceptible            6734
               Resistant              2740
               Intermediate            156
Name: count, dtype: int64


In [7]:
#dplicate
print("duplicate genom x antibiotic:", lab.duplicated(["genome_id", "antibiotic"]).sum())

duplicate genom x antibiotic: 9565


In [8]:
#metadate si coloana de an
meta = pd.read_csv("../data/raw/genome_meta_ecoli.tsv", sep="\t", dtype=str)
print(meta.shape)
[c for c in meta.columns if "year" in c or "date" in c or "country" in c]

(114985, 121)


['completion_date',
 'date_inserted',
 'date_modified',
 'isolation_country',
 'collection_date',
 'collection_year']

In [7]:
#imbinare si tabelul an × clasa (inlocuieste collection_year cu numele real din celula 6)
an = "collection_year"
cip = lab[lab["antibiotic"] == "ciprofloxacin"].merge(
    meta[["genome_id", an]], on="genome_id", how="left")
cip[an] = pd.to_numeric(cip[an], errors="coerce")
print("fara an:", cip[an].isna().sum(), "din", len(cip))
pd.crosstab(cip[an], cip["resistant_phenotype"])

fara an: 4812 din 16972


resistant_phenotype,Intermediate,Resistant,Susceptible
collection_year,,,
1970.0,0,0,1
1977.0,0,0,1
1994.0,0,0,1
1997.0,0,0,1
1998.0,0,0,3
1999.0,0,1,6
2000.0,0,2,2
2001.0,0,17,136
2002.0,1,22,270


In [9]:
#randuri fara fenotip
print(lab["resistant_phenotype"].isna().groupby(lab["antibiotic"]).agg(["sum", "size"]))
lab[lab["resistant_phenotype"].isna()][["measurement", "measurement_value", "laboratory_typing_method"]].head(10)

                sum   size
antibiotic                
ampicillin     5305  12873
ciprofloxacin  7342  16972


,measurement,measurement_value,laboratory_typing_method
10,>16,16,MIC
23,>32,32,MIC
103,6,6,Disk diffusion
134,0.008,0.008,MIC
250,>4,4,MIC
311,>4,4,MIC
321,0.008,0.008,MIC
336,36,36,Disk diffusion
430,34,34,Disk diffusion
464,<=0.25,0.25,MIC


In [10]:
#duplicate si contradictii
lab2 = lab.dropna(subset=["resistant_phenotype"]).copy()
g = lab2.groupby(["genome_id", "antibiotic"])["resistant_phenotype"]
print("perechi genom x antibiotic:", len(g.size()))
print("cu mai multe inregistrari:", (g.size() > 1).sum())
print("cu fenotipuri contradictorii:", (g.nunique() > 1).sum())

perechi genom x antibiotic: 15675
cu mai multe inregistrari: 1523
cu fenotipuri contradictorii: 41


In [11]:
#o eticheta per genom si antibiotic (contradictiile se elimina, intermediarele la fel)
eticheta = (lab2.groupby(["genome_id", "antibiotic"])["resistant_phenotype"]
            .agg(lambda s: s.iloc[0] if s.nunique() == 1 else "Conflict")
            .reset_index())
print(eticheta["resistant_phenotype"].value_counts())
curat = eticheta[eticheta["resistant_phenotype"].isin(["Resistant", "Susceptible"])].copy()

resistant_phenotype
Susceptible     8561
Resistant       6919
Intermediate     154
Conflict          41
Name: count, dtype: int64


In [12]:
#an, tara, prevalenta pe an
print("duplicate in metadate:", meta["genome_id"].duplicated().sum())
curat = curat.merge(meta[["genome_id", "collection_year", "isolation_country"]],
                    on="genome_id", how="left")
curat["collection_year"] = pd.to_numeric(curat["collection_year"], errors="coerce")
datat = curat.dropna(subset=["collection_year"])
print(datat.groupby("antibiotic")["genome_id"].nunique(), "genome unice cu an")

tab = pd.crosstab([datat["antibiotic"], datat["collection_year"]], datat["resistant_phenotype"])
tab["prevalenta_R"] = (tab["Resistant"] / (tab["Resistant"] + tab["Susceptible"])).round(2)
tab

duplicate in metadate: 0
antibiotic
ampicillin       5507
ciprofloxacin    6364
Name: genome_id, dtype: int64 genome unice cu an


resistant_phenotype            Resistant  Susceptible  prevalenta_R
antibiotic    collection_year                                      
ampicillin    1970.0                   1            0          1.00
              1977.0                   1            0          1.00
              1994.0                   0            1          0.00
              1997.0                   0            1          0.00
              1998.0                   2            1          0.67
...                                  ...          ...           ...
ciprofloxacin 2020.0                  23           83          0.22
              2021.0                   5           33          0.13
              2022.0                  22            4          0.85
              2023.0                  20            9          0.69
              2024.0                  43            9          0.83

[62 rows x 3 columns]

In [13]:
#tara si alte coloane utile
print(datat["isolation_country"].fillna("necunoscut").value_counts().head(8))
[c for c in meta.columns if any(k in c.lower() for k in
    ("mlst", "host", "source", "bioproject", "biosample", "contigs", "genome_length", "genome_quality"))]

isolation_country
Norway            6374
United Kingdom    1911
USA               1125
necunoscut         392
Thailand           289
Portugal           280
Germany            214
Pakistan           211
Name: count, dtype: int64


['bioproject_accession',
 'biosample_accession',
 'contigs',
 'genome_length',
 'genome_quality',
 'mlst',
 'cgmlst_id',
 'cgmlst_hc0',
 'cgmlst_hc2',
 'cgmlst_hc5',
 'cgmlst_hc10',
 'cgmlst_hc20',
 'cgmlst_hc50',
 'cgmlst_hc100',
 'host_name',
 'isolation_source',
 'host_common_name',
 'host_group',
 'host_scientific_name',
 'host_health',
 'genome_quality_flags',
 'host_age',
 'host_gender',
 'lab_host']

In [14]:
#genome unice si tari
d = datat.drop_duplicates(["genome_id", "antibiotic"])
print(d.groupby("antibiotic")["genome_id"].nunique())
d.groupby([d["isolation_country"].fillna("necunoscut"), "antibiotic"]).size().unstack().sort_values("ciprofloxacin", ascending=False).head(8)

antibiotic
ampicillin       5507
ciprofloxacin    6364
Name: genome_id, dtype: int64


antibiotic,ampicillin,ciprofloxacin
isolation_country,,
Norway,3218.0,3156.0
United Kingdom,422.0,1489.0
USA,582.0,543.0
necunoscut,196.0,196.0
Portugal,140.0,140.0
Thailand,150.0,139.0
Pakistan,108.0,103.0
India,80.0,80.0


In [15]:
#bioproiecte (sursa)
info = meta[["genome_id", "bioproject_accession", "mlst", "genome_quality", "contigs",
             "genome_length", "cgmlst_hc0", "cgmlst_hc5", "cgmlst_hc20", "cgmlst_hc100"]]
datat = datat.merge(info, on="genome_id", how="left")
for tara in ["Norway", "United Kingdom"]:
    print(tara)
    print(datat[datat["isolation_country"] == tara].drop_duplicates("genome_id")["bioproject_accession"].value_counts().head(5))

Norway
bioproject_accession
PRJNA349231    1
PRJNA389301    1
PRJNA278886    1
Name: count, dtype: int64
United Kingdom
bioproject_accession
PRJEB4681    1483
Name: count, dtype: int64


In [16]:
#Norvegia, ciprofloxacin, pe ani
no = datat[(datat["isolation_country"] == "Norway") & (datat["antibiotic"] == "ciprofloxacin")].drop_duplicates("genome_id")
t = pd.crosstab(no["collection_year"], no["resistant_phenotype"])
t["prevalenta_R"] = (t["Resistant"] / (t["Resistant"] + t["Susceptible"])).round(2)
t

resistant_phenotype,Resistant,Susceptible,prevalenta_R
collection_year,,,
1999.0,1,1,0.50
2000.0,2,1,0.67
2002.0,3,78,0.04
2003.0,2,117,0.02
2004.0,1,122,0.01
2005.0,9,126,0.07
2006.0,11,131,0.08
2007.0,8,133,0.06
2008.0,14,145,0.09


In [17]:
#acoperirea informatiei de linie si calitatea
print(no[["mlst", "cgmlst_hc0", "cgmlst_hc5", "cgmlst_hc20", "cgmlst_hc100"]].notna().mean().round(2))
print(no["mlst"].value_counts().head(8))
print(no["genome_quality"].value_counts(dropna=False))

mlst            1.0
cgmlst_hc0      1.0
cgmlst_hc5      1.0
cgmlst_hc20     1.0
cgmlst_hc100    1.0
dtype: float64
mlst
MLST.ecoli_achtman_4.73     501
MLST.ecoli_achtman_4.95     364
MLST.ecoli_achtman_4.69     254
MLST.ecoli_achtman_4.131    250
MLST.ecoli_achtman_4.12     134
MLST.ecoli_achtman_4.127    124
MLST.ecoli_achtman_4.141     78
MLST.ecoli_achtman_4.10      65
Name: count, dtype: int64
genome_quality
Good    3153
Poor       3
Name: count, dtype: int64


In [18]:
#verificam sursa norvegiana
cand = [c for c in meta.columns if any(k in c.lower() for k in
        ("public", "strain", "sequencing", "center", "comment", "sra", "assembly"))]
print(cand)
meta[meta["genome_id"].isin(no["genome_id"])][cand].notna().mean().round(2)

['assembly_accession', 'assembly_method', 'public', 'publication', 'sequencing_centers', 'sequencing_depth', 'sequencing_platform', 'strain', 'sequencing_status', 'comments', 'sra_accession', 'isolation_comments', 'type_strain']


assembly_accession     0.00
assembly_method        0.00
public                 1.00
publication            1.00
sequencing_centers     0.00
sequencing_depth       0.00
sequencing_platform    1.00
strain                 1.00
sequencing_status      1.00
comments               0.00
sra_accession          0.99
isolation_comments     0.00
type_strain            0.00
dtype: float64

In [19]:
nor = meta[meta["genome_id"].isin(no["genome_id"])]
print(nor["publication"].value_counts().head())
print(nor["sequencing_platform"].value_counts())

publication
38219757    3129
29112974      27
Name: count, dtype: int64
sequencing_platform
Illumina HiSeq    3129
Illumina MiSeq      27
Name: count, dtype: int64
